In [0]:
from pyspark.sql import functions as F
from datetime import timedelta

audit_table = "f1_catalog.gold.data_quality_audit"

# Only check failures from the latest DQ run (within 15 min of the most recent check)
latest_time = (
    spark.table(audit_table)
    .agg(F.max("check_timestamp"))
    .collect()[0][0]
)

window_start = latest_time - timedelta(minutes=5)

failed_count = (
    spark.table(audit_table)
    .filter(
        (F.col("check_status") == "FAIL")
        & (F.col("check_timestamp") >= F.lit(window_start))
    )
    .count()
)

print(f"Failed DQ checks: {failed_count}")

if failed_count > 0:
    raise Exception(
        f"Data Quality failed. "
        f"{failed_count} failed checks found."
    )

print("Data Quality PASSED.")